In [7]:
import requests
import json
import ray
import time


@ray.remote
def request(api):
    response = requests.get(api)
    return json.loads(response.text)

@ray.remote
def job():
    for i in range(1000):
        result = request.remote('http://kiototsdb.kweather.co.kr:24242/api/query?start=2022/07/23-00:00:00&end=2022/07/24-00:00:00&m=sum:kw-iaq-sensor-kiot.ICW0W2000070{sensor=pm10|pm25|co2|voc|temp|humi}')

if __name__ == '__main__':
    start_time = time.time()

    num_cpus = 48
    ray.init(num_cpus=num_cpus, ignore_reinit_error=True)
    
    workers = [job.remote() for _ in range(num_cpus)]
    
    ray.get(workers)
    ray.shutdown()
    
    
    elapsed_time = time.time() - start_time
    print(elapsed_time, '경과시간')

2023-06-21 13:54:12,859	INFO worker.py:1636 -- Started a local Ray instance.


11.014859676361084 경과시간


In [8]:
import requests
import json
from multiprocessing import Pool

def request(api):
    response = requests.get(api)
    return json.loads(response.text)

def job(_):
    for i in range(1000):
        result = request('http://kwall3.kweather.co.kr:30101/v1/sensors/kw-aws-min3')
        print(i, '성공')

if __name__ == '__main__':
    start_time = time.time()
    
    num_processes = 48
    pool = Pool(processes=num_processes)
    
    pool.map(job, range(num_processes))
    
    pool.close()
    pool.join()
    elapsed_time = time.time() - start_time
    print(elapsed_time, '경과시간')

In [6]:
import asyncio
import time
import aiohttp

async def request(api):
    async with aiohttp.ClientSession() as session:
        async with session.get(api) as response:
            return await response.json()

async def job():
    for i in range(1000):
        result = await request('http://kwall3.kweather.co.kr:30101/v1/sensors/kw-aws-min3')

async def measure_execution_time():
    start_time = time.time()
    await job()
    elapsed_time = time.time() - start_time
    print(f"Elapsed time: {elapsed_time} seconds")

# Check if an event loop is already running
if asyncio.get_event_loop().is_running():
    asyncio.run_coroutine_threadsafe(measure_execution_time(), asyncio.get_event_loop())
else:
    asyncio.run(measure_execution_time())

0 성공
1 성공
2 성공
3 성공
4 성공
5 성공
6 성공
7 성공
8 성공
9 성공
10 성공
11 성공
12 성공
13 성공
14 성공
15 성공
16 성공
17 성공
18 성공
19 성공
20 성공
21 성공
22 성공
23 성공
24 성공
25 성공
26 성공
27 성공
28 성공
29 성공
30 성공
31 성공
32 성공
33 성공
34 성공
35 성공
36 성공
37 성공
38 성공
39 성공
40 성공
41 성공
42 성공
43 성공
44 성공
45 성공
46 성공
47 성공
48 성공
49 성공
50 성공
51 성공
52 성공
53 성공
54 성공
55 성공
56 성공
57 성공
58 성공
59 성공
60 성공
61 성공
62 성공
63 성공
64 성공
65 성공
66 성공
67 성공
68 성공
69 성공
70 성공
71 성공
72 성공
73 성공
74 성공
75 성공
76 성공
77 성공
78 성공
79 성공
80 성공
81 성공
82 성공
83 성공
84 성공
85 성공
86 성공
87 성공
88 성공
89 성공
90 성공
91 성공
92 성공
93 성공
94 성공
95 성공
96 성공
97 성공
98 성공
99 성공
100 성공
101 성공
102 성공
103 성공
104 성공
105 성공
106 성공
107 성공
108 성공
109 성공
110 성공
111 성공
112 성공
113 성공
114 성공
115 성공
116 성공
117 성공
118 성공
119 성공
120 성공
121 성공
122 성공
123 성공
124 성공
125 성공
126 성공
127 성공
128 성공
129 성공
130 성공
131 성공
132 성공
133 성공
134 성공
135 성공
136 성공
137 성공
138 성공
139 성공
140 성공
141 성공
142 성공
143 성공
144 성공
145 성공
146 성공
147 성공
148 성공
149 성공
150 성공
151 성공
152 성공
153 성공
154 성공
155 성공
156 성공
157 성공
158 

In [15]:
import asyncio
import aiohttp
import time
import ray

@ray.remote
class RequestActor:
    async def request(self, api):
        async with aiohttp.ClientSession() as session:
            async with session.get(api) as response:
                return await response.json()

async def job():
    request_actor = RequestActor.remote()
    tasks = []
    for i in range(1000):
        task = request_actor.request.remote('http://kiototsdb.kweather.co.kr:24242/api/query?start=2022/07/23-00:00:00&end=2022/07/24-00:00:00&m=sum:kw-iaq-sensor-kiot.ICW0W2000070{sensor=pm10|pm25|co2|voc|temp|humi}')
        tasks.append(task)

    await asyncio.gather(*tasks)

def run_job():
    asyncio.run_coroutine_threadsafe(job(), asyncio.get_event_loop())

if __name__ == '__main__':
    start_time = time.time()

    num_cpus = 4
    ray.init(num_cpus=num_cpus, ignore_reinit_error=True)

    threads = [ray.remote(run_job).remote() for _ in range(num_cpus)]
    ray.get(threads)

    ray.shutdown()

    elapsed_time = time.time() - start_time
    print(elapsed_time, '경과시간')

2023-06-21 14:42:18,900	INFO worker.py:1636 -- Started a local Ray instance.


7.277385711669922 경과시간


In [16]:
import requests
import json
import ray
import time


def request(api):
    response = requests.get(api)
    return json.loads(response.text)

def job():
    for i in range(1000):
        result = request('http://kiototsdb.kweather.co.kr:24242/api/query?start=2022/07/23-00:00:00&end=2022/07/24-00:00:00&m=sum:kw-iaq-sensor-kiot.ICW0W2000070{sensor=pm10|pm25|co2|voc|temp|humi}')

if __name__ == '__main__':
    start_time = time.time()

    job()
    
    elapsed_time = time.time() - start_time
    print(elapsed_time, '경과시간')

89.92395234107971 경과시간
